# Instruction fine-tuning with NeMo Data Designer, NeMo Curator and NeMo Automodel (on a free-tier-sized T4)

**The idea:** teach a tiny model (Qwen2.5-0.5B-Instruct) a new *house style* using only synthetic data that we generate, clean and train on ourselves, with three NVIDIA NeMo tools:

| Stage | Tool | What it does here |
|---|---|---|
| Generate | **NeMo Data Designer** | Builds a table of questions and answers by calling a "teacher" LLM column by column |
| Clean | **NeMo Curator** | Filters out badly formatted / too short / duplicate rows with a pipeline of stages |
| Train | **NeMo Automodel** | Fine-tunes the student from one YAML recipe |

**The target behaviour:** every answer must be **exactly 3 bullet points ("- ...") followed by a one-line "TL;DR: ..."**. The base model never does this (0 of 30 test answers). After fine-tuning it does it on all 30 held-out questions.

**Roadmap**
1. Check the machine
2. Install the tools (each in its own isolated environment)
3. Start a local teacher LLM (Qwen2.5-3B-Instruct on vLLM, so no API key is needed)
4. Generate 800 raw examples with Data Designer
5. Curate them with Curator (800 -> 445 clean rows)
6. Measure the untouched student (the "before" number)
7. Fine-tune the student with Automodel
8. Measure again and compare before / after

**To reproduce:** use a **T4 + High-RAM** runtime and run the cells top to bottom (about 25 minutes in total).

## Step 1: Check the machine
Confirms we have a Tesla T4 (15 GB GPU memory, compute capability 7.5), about 50 GB of CPU RAM (High-RAM runtime), Python 3.13 and CUDA 13. The T4 has **no native bf16**, so everything is trained in float32 and we pick a small 0.5B-parameter student model.

In [ ]:
# STEP 1: check the machine (GPU, RAM, disk, Python, CUDA)
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv; free -h | head -2; df -h /content | tail -1; python --version; nvcc --version | tail -2

name, memory.total [MiB], driver_version, compute_cap
Tesla T4, 15360 MiB, 580.82.07, 7.5
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.3Gi        46Gi       2.4Mi       3.6Gi        49Gi
overlay         236G   43G  194G  19% /
Python 3.13.16
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


## Step 2: Install the tools, each in its own isolated environment
The four tools pin different versions of PyTorch and other libraries, so installing them together would clash. We use **uv** to create four separate Python 3.12 environments under `/content/envs` and install them in parallel:
- **vllm**: a fast inference server. It plays the "teacher" LLM that Data Designer calls (no API key needed, everything stays in Colab).
- **dd**: **NeMo Data Designer** (`data-designer`), generates the synthetic instruction data.
- **curator**: **NeMo Curator** (`nemo-curator[text_cpu]`), filters and cleans the generated data.
- **automodel**: **NeMo Automodel** (`nemo_automodel`), fine-tunes the student model.

In [ ]:
%%bash
# STEP 2: create 4 isolated envs and install each tool in parallel (takes several minutes)
pip -q install uv
mkdir -p /content/envs /content/work && cd /content/envs
for n in vllm dd curator automodel; do uv venv --python 3.12 -q $n; done
( uv pip install -q --python vllm/bin/python vllm > /content/work/inst_vllm.log 2>&1 && echo "vllm OK" || echo "vllm FAILED" ) &
( uv pip install -q --python dd/bin/python data-designer pandas pyarrow > /content/work/inst_dd.log 2>&1 && echo "data-designer OK" || echo "data-designer FAILED" ) &
( uv pip install -q --python curator/bin/python "nemo-curator[text_cpu]" > /content/work/inst_curator.log 2>&1 && echo "nemo-curator OK" || echo "nemo-curator FAILED" ) &
( uv pip install -q --python automodel/bin/python nemo_automodel > /content/work/inst_automodel.log 2>&1 && echo "nemo-automodel OK" || echo "nemo-automodel FAILED" ) &
wait
echo ALL_INSTALLS_FINISHED

data-designer OK
vllm OK
nemo-automodel OK
nemo-curator OK
ALL_INSTALLS_FINISHED


## Step 3: Start a local "teacher" LLM
NeMo Data Designer creates data by calling an LLM through an OpenAI-compatible API. Normally that is a hosted model (NVIDIA Build, OpenAI, etc.) and needs an API key. To keep this notebook self-contained, we serve **Qwen2.5-3B-Instruct** locally with vLLM on the T4 and point Data Designer at `http://localhost:8000/v1`.

- `--dtype half`: the T4 works best in float16.
- `--enforce-eager`: skips CUDA-graph warm-up to save time and memory.
- `--served-model-name teacher`: the name Data Designer will use.
The server runs in the background; the cell waits until it answers.

In [ ]:
%%bash
# STEP 3: serve the teacher model with vLLM (background) and wait until it is ready
cd /content/work
nohup /content/envs/vllm/bin/vllm serve Qwen/Qwen2.5-3B-Instruct --served-model-name teacher --port 8000 \
  --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --enforce-eager > /content/work/vllm.log 2>&1 &
for i in $(seq 1 90); do
  if curl -s localhost:8000/v1/models | grep -q teacher; then echo "vLLM READY after $((i*5))s"; break; fi
  sleep 5
done
curl -s localhost:8000/v1/models | head -c 300; echo
tail -3 /content/work/vllm.log

vLLM READY after 90s
{"object":"list","data":[{"id":"teacher","object":"model","created":1792798154,"owned_by":"vllm","root":"Qwen/Qwen2.5-3B-Instruct","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-828123db295e863a","object":"model_permission","created":1792798154,"allow_create_engine":false,"allow_s
INFO:     Application startup complete.
INFO:     127.0.0.1:55388 - "GET /v1/models HTTP/1.1" 200 OK
INFO:     127.0.0.1:55390 - "GET /v1/models HTTP/1.1" 200 OK


## Step 4: Generate instruction data with NeMo Data Designer
**What we are building:** a tiny "style" dataset that teaches a small model to answer in a fixed house format: exactly **3 bullet points followed by a one-line `TL;DR:`**.

Data Designer builds a table column by column, and later columns can use earlier ones through Jinja templates (`{{ topic }}`):
1. **topic** and **style** (samplers): randomly picked from lists, which gives variety without hand-writing questions.
2. **question** (LLM column): the teacher writes a realistic user question for that topic and style.
3. **answer** (LLM column): the teacher answers it, following a strict format in the system prompt.

Two model aliases are used so each column gets its own settings: a hot, creative one for questions and a cooler, precise one for answers. The answer prompt contains a **one-shot example** of the exact format. (A first attempt without the example produced the format in only 33 of 500 rows, about 7%, which is why the example is there. Even with it the teacher still slips sometimes, and **Step 5 (Curator) removes those rows**.)

The script first previews 3 rows, then generates 800.

In [ ]:
%%bash
# STEP 4: write + run the Data Designer script inside the isolated "dd" environment
rm -rf /content/work/dd_artifacts; mkdir -p /content/work /content/data
cat > /content/work/dd_generate.py <<'PY'
import pandas as pd
import data_designer.config as dd
from data_designer.interface import DataDesigner

# 1) Where the LLM lives: our local vLLM server (OpenAI-compatible, no real key needed)
provider = dd.ModelProvider(name="local-vllm", endpoint="http://localhost:8000/v1",
                            provider_type="openai", api_key="not-needed")

# 2) Two "model aliases" = the same teacher with different sampling settings
models = [
    dd.ModelConfig(alias="question-writer", model="teacher", provider="local-vllm",
        inference_parameters=dd.ChatCompletionInferenceParams(temperature=1.0, top_p=0.95, max_tokens=60, max_parallel_requests=16)),
    dd.ModelConfig(alias="answerer", model="teacher", provider="local-vllm",
        inference_parameters=dd.ChatCompletionInferenceParams(temperature=0.3, top_p=0.9, max_tokens=220, max_parallel_requests=16)),
]

TOPICS = ["personal budgeting", "healthy sleep habits", "home cooking basics", "online account security",
          "first aid basics", "time management", "recycling and the environment", "learning a new language",
          "beginner fitness", "public speaking", "vegetable gardening", "everyday physics",
          "planning a budget trip", "managing stress", "using spreadsheets", "caring for a pet"]
STYLES = ["a how-to question", "a why question", "a comparison question", "a request for beginner advice",
          "a quick factual question", "a 'what should I know about' question"]

ANSWER_SYSTEM = """You are a concise explainer. Reply in EXACTLY this format and nothing else: three lines that each start with "- ", then one final line that starts with "TL;DR:". Never write more than 3 bullets. Always finish with the TL;DR line.

Example
Question: Why do we need sleep?
Answer:
- Sleep lets the brain clear waste and consolidate memories.
- It gives the body time to repair muscles and tissues.
- Poor sleep weakens mood, focus and the immune system.
TL;DR: Sleep is how your brain and body recharge and repair."""

# 3) Describe the dataset column by column
b = dd.DataDesignerConfigBuilder(model_configs=models)
b.add_column(dd.SamplerColumnConfig(name="topic", sampler_type=dd.SamplerType.CATEGORY,
                                    params=dd.CategorySamplerParams(values=TOPICS)))
b.add_column(dd.SamplerColumnConfig(name="style", sampler_type=dd.SamplerType.CATEGORY,
                                    params=dd.CategorySamplerParams(values=STYLES)))
b.add_column(dd.LLMTextColumnConfig(name="question", model_alias="question-writer",
    system_prompt="You write realistic questions that everyday users ask an assistant.",
    prompt="Write ONE short question (at most 25 words) that a person might ask, as {{ style }}, about: {{ topic }}. Output only the question."))
b.add_column(dd.LLMTextColumnConfig(name="answer", model_alias="answerer",
    system_prompt=ANSWER_SYSTEM, prompt="{{ question }}"))

designer = DataDesigner(artifact_path="/content/work/dd_artifacts", model_providers=[provider])

# 4) Preview a few rows first (cheap sanity check), then generate the full dataset
prev = designer.preview(b, num_records=3)
for _, r in prev.dataset.iterrows():
    print("TOPIC:", r["topic"], "| STYLE:", r["style"]); print("Q:", r["question"]); print("A:", r["answer"]); print("-" * 40)

res = designer.create(b, num_records=800, dataset_name="instruct_raw")
df = res.load_dataset()
df.insert(0, "id", [f"row-{i}" for i in range(len(df))])
df.to_json("/content/data/raw_generated.jsonl", orient="records", lines=True, force_ascii=False)
print("GENERATED ROWS:", len(df))
PY
/content/envs/dd/bin/python /content/work/dd_generate.py 2>&1 | grep -v -E "^\s*$" | tail -60

[23:39:34] [INFO] 📊 Progress [128.4s]:
[23:39:34] [INFO]   |-- 🐔 column 'question': 800/800 (100%) 6.2 rec/s
[23:39:34] [INFO]   |-- 🚗 column 'answer': 582/800 (73%) 4.5 rec/s
[23:39:39] [INFO] 📊 Progress [133.5s]:
[23:39:39] [INFO]   |-- 🐔 column 'question': 800/800 (100%) 6.0 rec/s
[23:39:39] [INFO]   |-- ✈️ column 'answer': 614/800 (77%) 4.6 rec/s
[23:39:44] [INFO] 📊 Progress [138.5s]:
[23:39:44] [INFO]   |-- 🐔 column 'question': 800/800 (100%) 5.8 rec/s
[23:39:44] [INFO]   |-- ✈️ column 'answer': 639/800 (80%) 4.6 rec/s
[23:39:49] [INFO] 📊 Progress [143.5s]:
[23:39:49] [INFO]   |-- 🐔 column 'question': 800/800 (100%) 5.6 rec/s
[23:39:49] [INFO]   |-- ✈️ column 'answer': 671/800 (84%) 4.7 rec/s
[23:39:54] [INFO] 📊 Progress [148.6s]:
[23:39:54] [INFO]   |-- 🐔 column 'question': 800/800 (100%) 5.4 rec/s
[23:39:54] [INFO]   |-- ✈️ column 'answer': 701/800 (88%) 4.7 rec/s
[23:40:00] [INFO] 📊 Progress [153.6s]:
[23:40:00] [INFO]   |-- 🐔 column 'question': 800/800 (100%) 5.2 rec/s
[23:40:

## Step 5: Curate the data with NeMo Curator
The teacher does not always follow the format (we saw answers with 5 bullets, or with no TL;DR line). If we trained on those, the student would learn the mistakes too. **NeMo Curator** builds a *pipeline* of stages that read, score, filter and write data:

1. **JsonlReader** loads the generated rows.
2. **ScoreFilter + a custom `ThreeBulletsTldrFilter`**: keeps an answer only if it is exactly 3 "- " bullets followed by a `TL;DR:` line. This shows how to write your own Curator filter (`score_document` computes a score, `keep_document` decides).
3. **Built-in heuristic filters** (`WordCountFilter`, `NonAlphaNumericFilter`): remove answers that are too short/long or full of symbols, and questions that are too short/long.
4. **A custom dedup stage**: drops rows whose question is a repeat (after lower-casing and stripping punctuation), because duplicates make a model memorise instead of generalise.
5. **JsonlWriter** saves the survivors.

Afterwards the script splits the clean data into train / validation files for fine-tuning and holds out some questions for testing.

In [ ]:
%%bash
# STEP 5: curate the generated data inside the isolated "curator" environment
cat > /content/work/curate.py <<'PY'
import json, re, glob, random, dataclasses
from dataclasses import dataclass
import pandas as pd
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.base import ProcessingStage
from nemo_curator.tasks import DocumentBatch
from nemo_curator.stages.text.io.reader import JsonlReader
from nemo_curator.stages.text.io.writer import JsonlWriter
from nemo_curator.stages.text.filters import ScoreFilter, DocumentFilter
from nemo_curator.stages.text.filters.heuristic import WordCountFilter, NonAlphaNumericFilter

# ---- custom filter: the answer must be exactly 3 bullets + a TL;DR line ----
def follows_format(text):
    lines = [l.strip() for l in str(text).strip().split("\n") if l.strip()]
    return len(lines) == 4 and all(l.startswith("- ") for l in lines[:3]) and lines[3].startswith("TL;DR:") and len(lines[3]) > 10

class ThreeBulletsTldrFilter(DocumentFilter):
    def __init__(self):
        super().__init__(); self._name = "three_bullets_tldr"
    def score_document(self, text):
        return 1.0 if follows_format(text) else 0.0
    def keep_document(self, score):
        return score >= 1.0

# ---- custom stage: drop repeated questions ----
@dataclass
class DedupeByField(ProcessingStage[DocumentBatch, DocumentBatch]):
    field: str = "question"
    name: str = "dedupe_by_field"
    def inputs(self): return ["data"], [self.field]
    def outputs(self): return ["data"], [self.field]
    def process(self, batch):
        df = batch.to_pandas()
        key = df[self.field].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
        out = df[~key.duplicated(keep="first")]
        return dataclasses.replace(batch, data=out) if len(out) else None

raw = pd.read_json("/content/data/raw_generated.jsonl", lines=True)
print("RAW ROWS:", len(raw), "| answers that follow the format:", int(raw["answer"].map(follows_format).sum()))

pipe = Pipeline(name="instruction_curation", description="format + quality filters + dedup")
pipe.add_stage(JsonlReader(file_paths="/content/data/raw_generated.jsonl", files_per_partition=1,
                           fields=["id", "topic", "question", "answer"]))
# one ScoreFilter stage can hold several filters, each reading its own text field and writing its own score column
pipe.add_stage(ScoreFilter(
    filter_obj=[ThreeBulletsTldrFilter(),
                WordCountFilter(min_words=15, max_words=120),
                NonAlphaNumericFilter(max_non_alpha_numeric_to_text_ratio=0.3),
                WordCountFilter(min_words=4, max_words=40)],
    text_field=["answer", "answer", "answer", "question"],
    score_field=["format_ok", "answer_words", "answer_symbols", "question_words"]))
pipe.add_stage(DedupeByField(field="question"))
pipe.add_stage(JsonlWriter("/content/data/curated", mode="overwrite"))
res = pipe.run()

files = glob.glob("/content/data/curated/*.jsonl")
clean = pd.concat([pd.read_json(f, lines=True) for f in files]) if files else pd.DataFrame()
print("CURATED ROWS:", len(clean), "(kept %.0f%%)" % (100 * len(clean) / len(raw)))

# ---- split: train / validation for fine-tuning, plus held-out test questions ----
clean = clean.sample(frac=1.0, random_state=0).reset_index(drop=True)
test = clean.iloc[:30]; val = clean.iloc[30:50]; train = clean.iloc[50:]
cols = ["question", "answer"]
train[cols].to_json("/content/data/sft_train.jsonl", orient="records", lines=True, force_ascii=False)
val[cols].to_json("/content/data/sft_val.jsonl", orient="records", lines=True, force_ascii=False)
json.dump(test["question"].tolist(), open("/content/data/test_questions.json", "w"))
print("TRAIN/VAL/TEST:", len(train), len(val), len(test))
print("SAMPLE TRAIN ROW:\n", train.iloc[0]["question"], "\n", train.iloc[0]["answer"])
PY
/content/envs/curator/bin/python /content/work/curate.py 2>&1 | grep -E "RAW ROWS|CURATED|TRAIN/VAL|SAMPLE|^- |TL;DR|Error|error|Traceback|File \"" | tail -40

RAW ROWS: 800 | answers that follow the format: 567
CURATED ROWS: 445 (kept 56%)
TRAIN/VAL/TEST: 395 20 30
SAMPLE TRAIN ROW:
- Excel offers more advanced features and customization options.
- Google Sheets is free and accessible from any device via web browser.
TL;DR: Excel has more advanced features, while Google Sheets is free and convenient.


## Step 6: Free the GPU and measure the student *before* fine-tuning
We are done with the teacher, so we stop the vLLM server to give the GPU back. Then we test the **student**, Qwen2.5-0.5B-Instruct, on the 30 held-out questions. It gets the plain question with **no format instructions**, and we count how many answers happen to match the house format (exactly 3 "- " bullets + a `TL;DR:` line). This is the "before" number to beat.

In [ ]:
%%bash
# STEP 6: stop the teacher server, then evaluate the untouched student model
pkill -f "[v]llm serve"; sleep 5; nvidia-smi --query-gpu=memory.used --format=csv,noheader
cat > /content/work/eval.py <<'PY'
import sys, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model_path, tag = sys.argv[1], sys.argv[2]
def follows_format(text):
    lines = [l.strip() for l in str(text).strip().split("\n") if l.strip()]
    return len(lines) == 4 and all(l.startswith("- ") for l in lines[:3]) and lines[3].startswith("TL;DR:") and len(lines[3]) > 10
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float32).cuda().eval()
qs = json.load(open("/content/data/test_questions.json"))
outs = []
for q in qs:
    ids = tok.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        o = model.generate(**ids, max_new_tokens=160, do_sample=False)
    outs.append({"question": q, "answer": tok.decode(o[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()})
ok = sum(follows_format(r["answer"]) for r in outs)
json.dump(outs, open(f"/content/data/eval_{tag}.json", "w"), indent=1)
print(f"FORMAT COMPLIANCE [{tag}]: {ok}/{len(outs)} = {100*ok/len(outs):.0f}%")
for r in outs[:3]:
    print("Q:", r["question"]); print("A:", r["answer"]); print("-" * 30)
PY
/usr/bin/python3 /content/work/eval.py Qwen/Qwen2.5-0.5B-Instruct before 2>&1 | grep -v -i -E "warn|^\s*$" | tail -30

0 MiB
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 911.63it/s] 
FORMAT COMPLIANCE [before]: 0/30 = 0%
Q: Why do I need to prioritize costs for a budget trip?
A: Prioritizing costs for a budget trip is crucial because it helps you make informed decisions about your expenses and ensures that you allocate resources effectively. Here are some key reasons why prioritizing costs is important:
1. **Budgeting**: A well-organized budget allows you to track your spending in real-time, helping you understand where your money goes and how much you can realistically spend on each activity or destination.
2. **Avoiding Overruns**: By setting realistic expectations based on your available funds, you can avoid overspending on activities that exceed your budget. This prevents unexpected expenses and keeps your finances manageable.
3. **Flexibility**: Prioritizing costs gives you more flexibility when making travel plans. If unf

## Step 7: Instruction fine-tune the student with NeMo Automodel
**NeMo Automodel** fine-tunes Hugging Face models from a single YAML "recipe". Each block of the YAML is one decision:

- `model`: load Qwen2.5-0.5B-Instruct in **float32** (written `torch.float32`; the T4 has no bf16).
- `dataset` / `validation_dataset`: our curated JSONL files. `ColumnMappedTextInstructionDataset` maps our `question` and `answer` columns to prompt and response, and **`answer_only_loss_mask: true`** means the loss is computed only on the answer tokens, so the model learns to *answer*, not to repeat the question.
- `step_scheduler`: effective batch of 8 examples (2 per GPU pass, accumulated), about 150 steps (3 passes over the 395 examples), validation every 50 steps.
- **`use_hf_chat_template: true`**: formats each example with the model's own chat template, exactly like the prompt we use when testing. (Our first two attempts left this off, so the model trained on a plain-text layout and only partly carried the behaviour over to chat prompts.)
- `optimizer` / `lr_scheduler`: Adam with learning rate 5e-5 and cosine decay (full-weight fine-tuning). A first attempt with 1e-5 and 100 steps was **under-trained**: the model had only started adding a `TL;DR:` line and still failed every strict format check, so we raised the learning rate and trained longer.
- `checkpoint`: `save_consolidated: final` writes a normal Hugging Face folder at the end, so we can load it with transformers in Step 8.

Run it with one command: `automodel config.yaml`.

In [ ]:
%%bash
# STEP 7: write the Automodel recipe and launch full fine-tuning on the T4
rm -rf /content/work/sft_ckpt; mkdir -p /content/work/sft_ckpt
cat > /content/work/sft.yaml <<'YML'
recipe: TrainFinetuneRecipeForNextTokenPrediction

step_scheduler:            # how long to train and how often to validate / save
  global_batch_size: 8     # examples per optimizer step
  local_batch_size: 2      # examples per GPU pass (rest is gradient accumulation)
  num_epochs: 3
  max_steps: 150
  val_every_steps: 50
  ckpt_every_steps: 150

dist_env:
  backend: nccl
  timeout_minutes: 5

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1234
  ranked: true

model:                     # the student
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
  torch_dtype: torch.float32  # T4 has no native bf16

checkpoint:
  enabled: true
  checkpoint_dir: /content/work/sft_ckpt
  model_save_format: safetensors
  save_consolidated: final # also write a plain Hugging Face folder at the end

distributed:
  strategy: fsdp2
  dp_size: none
  tp_size: 1
  cp_size: 1
  sequence_parallel: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataset:                   # curated data from Step 5
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /content/data/sft_train.jsonl
  split: train
  column_mapping:
    question: question
    answer: answer
  seq_length: 384
  answer_only_loss_mask: true   # learn only from the answer tokens
  use_hf_chat_template: true    # format examples with the model's own chat template (same as at test time)
  padding: do_not_pad
  truncation: longest_first

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: true

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /content/data/sft_val.jsonl
  split: train
  column_mapping:
    question: question
    answer: answer
  seq_length: 384
  answer_only_loss_mask: true
  use_hf_chat_template: true
  padding: do_not_pad
  truncation: longest_first

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater

optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1.0e-8
  lr: 5.0e-5
  weight_decay: 0

lr_scheduler:
  lr_decay_style: cosine
  min_lr: 5.0e-6
YML
cd /content/work
/content/envs/automodel/bin/automodel /content/work/sft.yaml > /content/work/sft.log 2>&1
echo "EXIT CODE: $?"
# show a few progress lines, validation losses and where checkpoints were written
grep -E "\\| step (1|50|100|149) \\|" /content/work/sft.log | cut -c1-170
grep -E "val_loss|Traceback|Error" /content/work/sft.log | cut -c1-200 | tail -8
ls /content/work/sft_ckpt

EXIT CODE: 0
Training:   1%|          | 1/150 [00:04<11:22,  4.58s/step, loss=1.8555, lr=8.00e-
2026-10-24 00:05:59 | INFO | root | step 50 | epoch 1 | loss 0.5673 | grad_norm 8.0043 | lr 4.26e-05 | mem 9.23 GiB | tps 28.88(28.88/gpu) | num_label_tokens 511
2026-10-24 00:07:14 | INFO | root | step 100 | epoch 2 | loss 0.2070 | grad_norm 4.8924 | lr 1.81e-05 | mem 9.23 GiB | tps 37.61(37.61/gpu) | num_label_tokens 592
Training:  99%|█████████▉| 149/1
Training: 100%|██████████| 150/1
2026-10-24 00:05:58 | INFO | nemo_automodel.components.checkpoint.lifecycle | Updated LOWEST_VAL checkpoint symlink to epoch_0_step_49 (val_loss=1.4554)
epoch_0_step_49
epoch_1_step_99
epoch_2_step_149
LATEST
LOWEST_VAL
training.jsonl
validation.jsonl


## Step 8: Measure the student *after* fine-tuning
We load the fine-tuned weights (the `consolidated` Hugging Face folder Automodel wrote), ask the **same 30 held-out questions with the same plain prompt**, and count format compliance again. Then we print a few before/after answers side by side.

The test questions were never used for training, so a high score means the model learned the *behaviour*, not just memorised answers.

In [ ]:
%%bash
# STEP 8: evaluate the fine-tuned model and compare with the "before" run from Step 6
CKPT=$(ls -d /content/work/sft_ckpt/epoch_*_step_* | sort -V | tail -1)/model/consolidated
echo "using checkpoint: $CKPT"
/usr/bin/python3 /content/work/eval.py $CKPT after 2>&1 | grep "FORMAT COMPLIANCE"
/usr/bin/python3 - <<'EOF'
import json
def lines(t): return [l.strip() for l in t.strip().split("\n") if l.strip()]
def has_tldr(t): return any(l.startswith("TL;DR:") for l in lines(t))
def uses_dash_bullets(t): return sum(l.startswith("- ") for l in lines(t)) == 3
b = json.load(open("/content/data/eval_before.json")); a = json.load(open("/content/data/eval_after.json"))
for name, r in [("BEFORE", b), ("AFTER ", a)]:
    n = len(r)
    print(f"{name}: has TL;DR line {sum(has_tldr(x['answer']) for x in r)}/{n} | exactly 3 '- ' bullets {sum(uses_dash_bullets(x['answer']) for x in r)}/{n} | avg words {sum(len(x['answer'].split()) for x in r)/n:.0f}")
print()
for i in [0, 1, 2]:
    print("QUESTION:", b[i]["question"])
    print("--- BEFORE (base Qwen2.5-0.5B-Instruct) ---"); print(b[i]["answer"][:420])
    print("--- AFTER (fine-tuned with Automodel) ---"); print(a[i]["answer"])
    print("=" * 70)
EOF

using checkpoint: /content/work/sft_ckpt/epoch_2_step_149/model/consolidated
FORMAT COMPLIANCE [after]: 30/30 = 100%
BEFORE: has TL;DR line 0/30 | exactly 3 '- ' bullets 0/30 | avg words 119
AFTER : has TL;DR line 30/30 | exactly 3 '- ' bullets 30/30 | avg words 45

QUESTION: Why do I need to prioritize costs for a budget trip?
--- BEFORE (base Qwen2.5-0.5B-Instruct) ---
Prioritizing costs for a budget trip is crucial because it helps you make informed decisions about your expenses and ensures that you allocate resources effectively. Here are some key reasons why prioritizing costs is important:

1. **Budgeting**: A well-organized budget allows you to track your spending in real-time, helping you understand where your money goes and how much you can realistically spend on each activi
--- AFTER (fine-tuned with Automodel) ---
- Prioritizing costs ensures you stay within your travel budget.
- It helps maximize your enjoyment by choosing the best experiences.
- Allocating funds wisely pre

## Results and what to take away

| | Base student | Fine-tuned student |
|---|---|---|
| Answers matching the exact format (3 "- " bullets + TL;DR), 30 held-out questions | **0 / 30** | **30 / 30** |
| Answers with a TL;DR line | 0 / 30 | 30 / 30 |
| Average answer length | 119 words | 45 words |

**The data funnel:** Data Designer produced 800 rows, 567 of them already in the right format, and Curator's format, length, symbol and duplicate filters left **445 clean rows** (395 train, 20 validation, 30 held-out test). Without the one-shot example in the answer prompt, only 33 of 500 rows came out in the right format, so prompt design matters as much as filtering.

**Three lessons from getting here**
1. **Match training and test formatting.** With Automodel's default plain-text layout the model only partly learned the format when prompted through the chat template (0/30 strict, though it began adding TL;DR lines). Setting `use_hf_chat_template: true` fixed it.
2. **Curate before you train.** The Curator step is what guarantees every training answer follows the rule the model should learn.
3. **Keep expectations honest.** This measures *format following* on in-domain questions, using only 30 test questions. It says nothing about factual accuracy, and the 20-row validation loss was lowest after the first epoch (1.46), a hint that more data would help more than more epochs.

**Where things are:** curated data in `/content/data`, fine-tuned weights in `/content/work/sft_ckpt/epoch_2_step_149/model/consolidated`, per-question outputs in `/content/data/eval_before.json` and `eval_after.json`. They disappear when the runtime is deleted, so download what you want to keep.